# 05 — Federated training

**Purpose.** FedAvg, FedProx, SCAFFOLD and FedNova on a Dirichlet label-skew federation.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/05_federated_training_*.png|pdf` and `reports/notebooks/05_federated_training.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "05_federated_training"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Train four algorithms (virtual clients, one process)

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.data_partitioning import partition
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig
d = make_dataset(n_samples=6000, seed=SEED); parts = partition("dirichlet", d.y_train, 10, SEED, alpha=.5)
res = {a: run_fl(FLConfig(algo=a, rounds=15, local_epochs=2, mu=.05 if a == "fedprox" else 0., seed=SEED), d, parts) for a in ("fedavg", "fedprox", "scaffold", "fednova")}
tab = pd.DataFrame([dict(algo=a, acc=r.final["acc"], f1=r.final["f1"], auc=r.final["auc"], up_mb=r.totals.get("up_mb")) for a, r in res.items()]).round(4); display(tab)
SUMMARY["final"] = tab.to_dict("records")

,algo,acc,f1,auc,up_mb
0,fedavg,0.8944,0.8586,0.9586,None
1,fedprox,0.8883,0.8508,0.9553,None
2,scaffold,0.9044,0.8827,0.9627,None
3,fednova,0.9033,0.8790,0.9639,None


## 2. Figure: convergence

In [3]:
fig, ax = plt.subplots(figsize=(4.6, 3))
for a, r in res.items(): ax.plot([h["acc"] for h in r.history], label=a)
ax.set_xlabel("round"); ax.set_ylabel("test accuracy"); ax.legend(); save_fig(fig, "convergence"); save_summary()

summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/05_federated_training.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.